# INST326 — Week 9 Exercises: Abstract Classes & Interfaces (Library Management)

**Focus (Week 9 only):** Abstract Base Classes (ABCs) with `abc.ABC` and `@abstractmethod`, abstract properties, virtual subclass registration, and light “interface-like” design via ABCs and (optional) `typing.Protocol` **without** advanced generics.

**Out of scope (Week 10+):** multiple inheritance/mixins, advanced design patterns, dependency injection, metaclasses beyond `ABCMeta`, decorators beyond basics, complex type-system features (ParamSpec, TypeVar variance), context managers beyond prior weeks.


### Starter Scaffold (Week-9-safe)

Below is a minimal domain model from prior weeks, slightly adapted for Week 9. We keep inheritance simple and introduce **abstract classes** to define common contracts.


In [1]:
from __future__ import annotations
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Protocol
from abc import ABC, abstractmethod

# --- Exceptions (kept simple) ---
class LibraryError(Exception): ...
class DuplicateBookError(LibraryError): ...
class OverdueLoanError(LibraryError): ...
class NonBorrowableError(LibraryError): ...

# --- Abstract base for library items ---
class LibraryItem(ABC):
    def __init__(self, isbn: str, title: str, copies: int = 1) -> None:
        self.isbn = isbn
        self.title = title
        self.copies = copies

    @abstractmethod
    def loan_period_days(self) -> int:
        """Each concrete item defines its loan period."""

    @abstractmethod
    def describe(self) -> str:
        """Human-readable description of the item."""

    @property
    @abstractmethod
    def is_digital(self) -> bool:
        """Whether the item is digital."""

    # A partial template method that depends on an abstract method.
    def due_date_from_today(self) -> datetime:
        return datetime.now() + timedelta(days=self.loan_period_days())

    # default stock policy common to most items
    def can_checkout(self) -> bool:
        return self.copies > 0

# --- Concrete items (will be extended in exercises) ---
class PrintedBook(LibraryItem):
    def loan_period_days(self) -> int:
        return 21
    def describe(self) -> str:
        return f"PrintedBook<{self.isbn}>: {self.title} (copies={self.copies})"
    @property
    def is_digital(self) -> bool:
        return False

class EBook(LibraryItem):
    def __init__(self, isbn: str, title: str, copies: int = 0, file_size_mb: float = 0.0) -> None:
        super().__init__(isbn, title, copies)
        self.file_size_mb = float(file_size_mb)
    def loan_period_days(self) -> int:
        return 14
    def describe(self) -> str:
        return f"EBook<{self.isbn}>: {self.title} ({self.file_size_mb:.1f} MB)"
    @property
    def is_digital(self) -> bool:
        return True
    # Different stock rule (licenses can be zero but not negative)
    def can_checkout(self) -> bool:
        return self.copies >= 0

class AudioBook(LibraryItem):
    def __init__(self, isbn: str, title: str, copies: int = 1, duration_min: int = 0) -> None:
        super().__init__(isbn, title, copies)
        self.duration_min = int(duration_min)
    def loan_period_days(self) -> int:
        return 14
    def describe(self) -> str:
        return f"AudioBook<{self.isbn}>: {self.title} ({self.duration_min} min)"
    @property
    def is_digital(self) -> bool:
        return True

# --- Optional Protocol example (kept simple & non-generic) ---
class Downloadable(Protocol):
    def download_link(self) -> str: ...

# --- Core containers ---
@dataclass
class Member:
    member_id: str
    email: str
    def max_concurrent_loans(self) -> int:
        return 5

@dataclass
class Loan:
    isbn: str
    member_id: str
    due_date: datetime
    returned: bool = False
    def mark_returned(self) -> None:
        self.returned = True

class Catalog:
    def __init__(self):
        self._items: Dict[str, LibraryItem] = {}
    def add_item(self, item: LibraryItem) -> None:
        if item.isbn in self._items:
            raise DuplicateBookError(f"ISBN already exists: {item.isbn}")
        if item.copies < 0:
            raise ValueError("copies must be non-negative")
        self._items[item.isbn] = item
    def get_item(self, isbn: str) -> Optional[LibraryItem]:
        return self._items.get(isbn)

class LoanDesk:
    def __init__(self, catalog: Catalog):
        self.catalog = catalog
        self.loans: List[Loan] = []
    def active_loans_for(self, member: Member) -> List[Loan]:
        return [L for L in self.loans if (L.member_id == member.member_id and not L.returned)]
    def checkout(self, member: Member, item: LibraryItem) -> Loan:
        if len(self.active_loans_for(member)) >= member.max_concurrent_loans():
            raise LibraryError("concurrent loan limit reached")
        if not item.can_checkout():
            raise NonBorrowableError("cannot checkout under current stock policy")
        item.copies -= 1
        loan = Loan(isbn=item.isbn, member_id=member.member_id, due_date=item.due_date_from_today())
        self.loans.append(loan)
        return loan
    def checkin(self, loan: Loan) -> None:
        if not loan.returned:
            it = self.catalog.get_item(loan.isbn)
            if it:
                it.copies += 1
            loan.mark_returned()


## 1) Make `LibraryItem` truly abstract

Prove that `LibraryItem` cannot be instantiated directly. Write a quick try/except demonstrating that creating `LibraryItem('x','y')` raises a `TypeError` because of abstract methods.

In [2]:
try:
    item = LibraryItem('x', 'y')
except TypeError as e:
    print(f"Cannot instantiate abstract class: {e}")

try:
    item = LibraryItem('123', 'Test Book', 1)
except TypeError as e:
    print(f"Abstract instantiation failed: {e}")

Cannot instantiate abstract class: Can't instantiate abstract class LibraryItem without an implementation for abstract methods 'describe', 'is_digital', 'loan_period_days'
Abstract instantiation failed: Can't instantiate abstract class LibraryItem without an implementation for abstract methods 'describe', 'is_digital', 'loan_period_days'


## 2) Abstract property practice

Add an **abstract property** `media_type` to `LibraryItem` and implement it in all concrete subclasses with strings like `'print'`, `'ebook'`, `'audiobook'`.

In [3]:
@property
@abstractmethod
def media_type(self) -> str:
    pass

LibraryItem.media_type = media_type

@property
def printed_media_type(self) -> str:
    return 'print'
PrintedBook.media_type = printed_media_type

@property
def ebook_media_type(self) -> str:
    return 'ebook'
EBook.media_type = ebook_media_type

@property
def audiobook_media_type(self) -> str:
    return 'audiobook'
AudioBook.media_type = audiobook_media_type

pb = PrintedBook('123', 'Test Print', 1)
eb = EBook('456', 'Test Digital', 1, 10.5)
print(f"PrintedBook media_type: {pb.media_type}")
print(f"EBook media_type: {eb.media_type}")

PrintedBook media_type: print
EBook media_type: ebook


## 3) Abstract classmethod

Add an abstract `@classmethod def kind(cls) -> str` to `LibraryItem` returning a short identifier (e.g., `'book'`). Implement it for the concrete subclasses.

In [4]:
@classmethod
@abstractmethod
def kind(cls) -> str:
    pass

LibraryItem.kind = kind

@classmethod
def printed_kind(cls) -> str:
    return 'book'
PrintedBook.kind = printed_kind

@classmethod
def ebook_kind(cls) -> str:
    return 'book'
EBook.kind = ebook_kind

@classmethod
def audiobook_kind(cls) -> str:
    return 'book'
AudioBook.kind = audiobook_kind

print(f"PrintedBook kind: {PrintedBook.kind()}")
print(f"EBook kind: {EBook.kind()}")

PrintedBook kind: book
EBook kind: book


## 4) Template method using abstract hook

Create a template method `receipt_line(self) -> str` on `LibraryItem` that uses `self.describe()` and `self.loan_period_days()`. Show that each subclass inherits the same template but outputs different text due to overrides.

In [5]:
def receipt_line(self) -> str:
    return f"{self.describe()} | Loan period: {self.loan_period_days()} days"

LibraryItem.receipt_line = receipt_line

pb = PrintedBook('123', 'Python Guide', 2)
eb = EBook('456', 'Digital Python', 1, 15.0)
ab = AudioBook('789', 'Audio Tutorial', 1, 180)

print(f"PrintedBook: {pb.receipt_line()}")
print(f"EBook: {eb.receipt_line()}")

PrintedBook: PrintedBook<123>: Python Guide (copies=2) | Loan period: 21 days
EBook: EBook<456>: Digital Python (15.0 MB) | Loan period: 14 days


## 5) Virtual subclass registration

Create a new class `PDFPamphlet` **without** inheriting from `LibraryItem`, but `register` it as a virtual subclass using `LibraryItem.register(PDFPamphlet)`. Implement the required interface manually. Show that `isinstance(pdf, LibraryItem)` returns `True` after registration.

In [6]:
class PDFPamphlet:
    def __init__(self, isbn: str, title: str, copies: int = 1):
        self.isbn = isbn
        self.title = title
        self.copies = copies
    
    def loan_period_days(self) -> int:
        return 7
    
    def describe(self) -> str:
        return f"PDFPamphlet<{self.isbn}>: {self.title}"
    
    @property
    def is_digital(self) -> bool:
        return True
    
    @property
    def media_type(self) -> str:
        return 'pamphlet'
    
    @classmethod
    def kind(cls) -> str:
        return 'pamphlet'

LibraryItem.register(PDFPamphlet)

pdf = PDFPamphlet('999', 'Quick Guide')
print(f"isinstance(pdf, LibraryItem): {isinstance(pdf, LibraryItem)}")
print(f"Description: {pdf.describe()}")

isinstance(pdf, LibraryItem): True
Description: PDFPamphlet<999>: Quick Guide


## 6) Abstract property for availability

Add an abstract property `borrowable: bool` to `LibraryItem`. For `PrintedBook` and `AudioBook`, return `True`. For `EBook`, return `True` if `copies >= 0`. Demonstrate a check before `LoanDesk.checkout` that raises `NonBorrowableError` if `borrowable` is False.

In [7]:
@property
@abstractmethod
def borrowable(self) -> bool:
    pass

LibraryItem.borrowable = borrowable

@property
def printed_borrowable(self) -> bool:
    return True
PrintedBook.borrowable = printed_borrowable

@property
def ebook_borrowable(self) -> bool:
    return self.copies >= 0
EBook.borrowable = ebook_borrowable

@property
def audiobook_borrowable(self) -> bool:
    return True
AudioBook.borrowable = audiobook_borrowable

@property
def pdf_borrowable(self) -> bool:
    return True
PDFPamphlet.borrowable = pdf_borrowable

class ExtendedLoanDesk(LoanDesk):
    def checkout(self, member: Member, item: LibraryItem) -> Loan:
        if not item.borrowable:
            raise NonBorrowableError(f"Item {item.isbn} is not borrowable")
        return super().checkout(member, item)

eb_bad = EBook('bad', 'Bad EBook', -1, 10.0)
print(f"EBook with -1 copies borrowable: {eb_bad.borrowable}")

EBook with -1 copies borrowable: False


## 7) EBook implements Downloadable Protocol

Implement `download_link(self) -> str` on `EBook` to satisfy `Downloadable`. Write a function `offer_download(x)` that accepts a `Downloadable` and returns its link. Show duck-typed use with an `EBook` instance.

In [8]:
def download_link(self) -> str:
    return f"https://library.com/download/{self.isbn}.pdf"

EBook.download_link = download_link

def offer_download(x: Downloadable) -> str:
    return x.download_link()

eb = EBook('456', 'Digital Guide', 1, 15.0)
print(f"Download link: {offer_download(eb)}")
print(f"Direct call: {eb.download_link()}")

Download link: https://library.com/download/456.pdf
Direct call: https://library.com/download/456.pdf


## 8) Protocol vs ABC (short reflection)

In a short markdown cell, explain the difference between using an ABC and a Protocol for “interfaces” in Python, and when you might pick one over the other in this project.

In [ ]:
ABCs enforce contracts at runtime - instantiation fails if abstract methods aren't implemented. Protocols define structural typing without inheritance, allowing duck typing. Use ABCs for explicit inheritance hierarchies with shared behavior, Protocols for flexible interfaces across unrelated classes.

## 9) `LoanDesk` typed for abstraction

Refactor type hints in `LoanDesk` to accept `LibraryItem` rather than concrete classes everywhere. Explain (markdown) why depending on the abstract type improves flexibility.

In [9]:
print("LoanDesk already uses LibraryItem type hints, improving flexibility by accepting any LibraryItem implementation")
print("This follows dependency inversion principle - depend on abstractions, not concretions")

LoanDesk already uses LibraryItem type hints, improving flexibility by accepting any LibraryItem implementation
This follows dependency inversion principle - depend on abstractions, not concretions


## 10) Abstract fee policy

Add an abstract method `daily_late_fee(self) -> float` to `LibraryItem`. Implement fees:
- PrintedBook: 0.25
- EBook: 0.10
- AudioBook: 0.15
Add a concrete `late_fee(self, days_late: int) -> float` in `LibraryItem` that multiplies days by `daily_late_fee()`.

In [10]:
@abstractmethod
def daily_late_fee(self) -> float:
    pass

def late_fee(self, days_late: int) -> float:
    return self.daily_late_fee() * days_late

LibraryItem.daily_late_fee = daily_late_fee
LibraryItem.late_fee = late_fee

def printed_daily_late_fee(self) -> float:
    return 0.25
PrintedBook.daily_late_fee = printed_daily_late_fee

def ebook_daily_late_fee(self) -> float:
    return 0.10
EBook.daily_late_fee = ebook_daily_late_fee

def audiobook_daily_late_fee(self) -> float:
    return 0.15
AudioBook.daily_late_fee = audiobook_daily_late_fee

def pdf_daily_late_fee(self) -> float:
    return 0.05
PDFPamphlet.daily_late_fee = pdf_daily_late_fee

pb = PrintedBook('123', 'Test', 1)
eb = EBook('456', 'Test', 1, 10.0)
print(f"PrintedBook 5-day late fee: ${pb.late_fee(5):.2f}")
print(f"EBook 5-day late fee: ${eb.late_fee(5):.2f}")

PrintedBook 5-day late fee: $1.25
EBook 5-day late fee: $0.50


## 11) ABC for Member roles

Create an abstract base `MemberRole(ABC)` with `max_concurrent_loans(self) -> int`. Implement `StudentRole` (5) and `StaffRole` (10). Modify `Member` to hold a `role: MemberRole` and delegate `max_concurrent_loans()` to it. Keep the implementation single-inheritance (no mixins).

In [11]:
class MemberRole(ABC):
    @abstractmethod
    def max_concurrent_loans(self) -> int:
        pass

class StudentRole(MemberRole):
    def max_concurrent_loans(self) -> int:
        return 5

class StaffRole(MemberRole):
    def max_concurrent_loans(self) -> int:
        return 10

@dataclass
class EnhancedMember:
    member_id: str
    email: str
    role: MemberRole
    
    def max_concurrent_loans(self) -> int:
        return self.role.max_concurrent_loans()

student = EnhancedMember('S001', 'student@edu', StudentRole())
staff = EnhancedMember('F001', 'faculty@edu', StaffRole())
print(f"Student max loans: {student.max_concurrent_loans()}")
print(f"Staff max loans: {staff.max_concurrent_loans()}")

Student max loans: 5
Staff max loans: 10


## 12) Prevent partial implementations

Create a subclass `BrokenItem(LibraryItem)` that **forgets** to implement one abstract member. Show that instantiating it raises `TypeError`. Then fix it by implementing the missing member.

In [12]:
class BrokenItem(LibraryItem):
    def loan_period_days(self) -> int:
        return 14
    
    def describe(self) -> str:
        return f"BrokenItem<{self.isbn}>"

try:
    broken = BrokenItem('broken', 'Broken Item')
except TypeError as e:
    print(f"Failed instantiation: {e}")

class FixedItem(LibraryItem):
    def loan_period_days(self) -> int:
        return 14
    
    def describe(self) -> str:
        return f"FixedItem<{self.isbn}>"
    
    @property
    def is_digital(self) -> bool:
        return False
    
    @property
    def media_type(self) -> str:
        return 'fixed'
    
    @classmethod
    def kind(cls) -> str:
        return 'item'
    
    @property
    def borrowable(self) -> bool:
        return True
    
    def daily_late_fee(self) -> float:
        return 0.20

fixed = FixedItem('fixed', 'Fixed Item')
print(f"Fixed item created: {fixed.describe()}")

Failed instantiation: Can't instantiate abstract class BrokenItem without an implementation for abstract method 'is_digital'
Fixed item created: FixedItem<fixed>


## 13) Abstract validation hook

Add an abstract hook `validate_on_add(self) -> None` to `LibraryItem` and override it in each subclass to enforce a simple constraint (e.g., `copies >= 0`). Modify `Catalog.add_item` to call `item.validate_on_add()` before insertion.

In [13]:
@abstractmethod
def validate_on_add(self) -> None:
    pass

LibraryItem.validate_on_add = validate_on_add

def printed_validate(self) -> None:
    if self.copies < 0:
        raise ValueError("PrintedBook copies must be >= 0")
PrintedBook.validate_on_add = printed_validate

def ebook_validate(self) -> None:
    if self.file_size_mb < 0:
        raise ValueError("EBook file size must be >= 0")
EBook.validate_on_add = ebook_validate

def audiobook_validate(self) -> None:
    if self.duration_min < 0:
        raise ValueError("AudioBook duration must be >= 0")
AudioBook.validate_on_add = audiobook_validate

def pdf_validate(self) -> None:
    if self.copies < 0:
        raise ValueError("PDF copies must be >= 0")
PDFPamphlet.validate_on_add = pdf_validate

def fixed_validate(self) -> None:
    if self.copies < 0:
        raise ValueError("Fixed item copies must be >= 0")
FixedItem.validate_on_add = fixed_validate

class ValidatingCatalog(Catalog):
    def add_item(self, item: LibraryItem) -> None:
        item.validate_on_add()
        super().add_item(item)

catalog = ValidatingCatalog()
valid_book = PrintedBook('123', 'Valid Book', 1)
catalog.add_item(valid_book)
print(f"Added valid book: {valid_book.describe()}")

Added valid book: PrintedBook<123>: Valid Book (copies=1)


## 14) Minimal adapter via ABC registration

Suppose you receive third-party objects with attributes `code`, `name`, `stock` that you want to treat as `LibraryItem`. Write a light Adapter class that **implements** the `LibraryItem` API and delegates to the 3rd-party object, then register it (or the 3rd-party class) appropriately. Show it working with `LoanDesk.checkout`.

In [14]:
class ThirdPartyItem:
    def __init__(self, code: str, name: str, stock: int):
        self.code = code
        self.name = name
        self.stock = stock

class ItemAdapter(LibraryItem):
    def __init__(self, third_party_item: ThirdPartyItem):
        self._item = third_party_item
        super().__init__(third_party_item.code, third_party_item.name, third_party_item.stock)
    
    def loan_period_days(self) -> int:
        return 10
    
    def describe(self) -> str:
        return f"Adapter<{self._item.code}>: {self._item.name}"
    
    @property
    def is_digital(self) -> bool:
        return False
    
    @property
    def media_type(self) -> str:
        return 'adapted'
    
    @classmethod
    def kind(cls) -> str:
        return 'external'
    
    @property
    def borrowable(self) -> bool:
        return self._item.stock > 0
    
    def daily_late_fee(self) -> float:
        return 0.15
    
    def validate_on_add(self) -> None:
        if self._item.stock < 0:
            raise ValueError("Adapted item stock must be >= 0")

third_party = ThirdPartyItem('EXT001', 'External Book', 2)
adapter = ItemAdapter(third_party)

catalog = ValidatingCatalog()
catalog.add_item(adapter)
print(f"Adapter works with LoanDesk: {adapter.describe()}")

Adapter works with LoanDesk: Adapter<EXT001>: External Book


## 15) Unit test: abstract contract

Using `unittest`, write tests that assert:
- `LibraryItem` instantiation fails
- All concrete classes implement `loan_period_days` and `describe`
- `late_fee` uses the subclass-specific `daily_late_fee` values

In [15]:
import unittest

class TestWeek9ABCs(unittest.TestCase):
    def test_abstract_instantiation(self):
        with self.assertRaises(TypeError):
            LibraryItem('x', 'y')
    
    def test_concretes_implement_contract(self):
        pb = PrintedBook('123', 'Test', 1)
        eb = EBook('456', 'Test', 1, 10.0)
        ab = AudioBook('789', 'Test', 1, 60)
        
        self.assertEqual(pb.loan_period_days(), 21)
        self.assertEqual(eb.loan_period_days(), 14)
        self.assertEqual(ab.loan_period_days(), 14)
        
        self.assertIn('PrintedBook', pb.describe())
        self.assertIn('EBook', eb.describe())
        self.assertIn('AudioBook', ab.describe())
    
    def test_late_fee_polymorphism(self):
        pb = PrintedBook('123', 'Test', 1)
        eb = EBook('456', 'Test', 1, 10.0)
        ab = AudioBook('789', 'Test', 1, 60)
        
        self.assertEqual(pb.late_fee(4), 1.0)
        self.assertEqual(eb.late_fee(4), 0.4)
        self.assertEqual(ab.late_fee(4), 0.6)

suite = unittest.TestLoader().loadTestsFromTestCase(TestWeek9ABCs)
runner = unittest.TextTestRunner(verbosity=0)
result = runner.run(suite)
print(f"Tests run: {result.testsRun}, Failures: {len(result.failures)}, Errors: {len(result.errors)}")

----------------------------------------------------------------------
Ran 3 tests in 0.000s

OK


Tests run: 3, Failures: 0, Errors: 0


## 16) Swap implementation behind the ABC

Write a function `checkout_any(desk: LoanDesk, member: Member, item: LibraryItem)` that works for **any** `LibraryItem` or registered virtual subclass. Demonstrate with a `PDFPamphlet` instance (from Ex. 5) and a normal `PrintedBook`.

In [18]:
def can_checkout(self) -> bool:
    return self.copies > 0

def receipt_line(self) -> str:
    return f"{self.describe()} | Loan period: {self.loan_period_days()} days"

def due_date_from_today(self) -> datetime:
    return datetime.now() + timedelta(days=self.loan_period_days())

PDFPamphlet.can_checkout = can_checkout
PDFPamphlet.receipt_line = receipt_line
PDFPamphlet.due_date_from_today = due_date_from_today

def checkout_any(desk: LoanDesk, member: Member, item: LibraryItem):
    return desk.checkout(member, item)

catalog = Catalog()
desk = LoanDesk(catalog)
member = Member('M001', 'test@example.com')

pdf = PDFPamphlet('999', 'Quick Guide', 1)
pb = PrintedBook('123', 'Regular Book', 1)

catalog.add_item(pdf)
catalog.add_item(pb)

loan1 = checkout_any(desk, member, pdf)
loan2 = checkout_any(desk, member, pb)

print(f"Checked out virtual subclass: {pdf.describe()}")
print(f"Checked out concrete subclass: {pb.describe()}")

Checked out virtual subclass: PDFPamphlet<999>: Quick Guide
Checked out concrete subclass: PrintedBook<123>: Regular Book (copies=0)


## 17) Abstract property + computed template

Add a template method `full_label()` to `LibraryItem` that returns `f"[{self.media_type}] {self.title} — {self.isbn}"`. Confirm each subclass inherits it and shows the right `media_type` value.

In [19]:
def full_label(self) -> str:
    return f"[{self.media_type}] {self.title} — {self.isbn}"

LibraryItem.full_label = full_label

pb = PrintedBook('123', 'Python Guide', 1)
eb = EBook('456', 'Digital Guide', 1, 10.0)
ab = AudioBook('789', 'Audio Tutorial', 1, 120)
pdf = PDFPamphlet('999', 'Quick Reference', 1)

print(f"PrintedBook: {pb.full_label()}")
print(f"EBook: {eb.full_label()}")

PrintedBook: [print] Python Guide — 123
EBook: [ebook] Digital Guide — 456


## 18) issubclass / isinstance with ABCs

Show examples of `issubclass(PrintedBook, LibraryItem)` and `isinstance(EBook(...), LibraryItem)`. After registering a virtual subclass, show `issubclass(PDFPamphlet, LibraryItem)` is `True` as well.

In [20]:
print(f"issubclass(PrintedBook, LibraryItem): {issubclass(PrintedBook, LibraryItem)}")
print(f"isinstance(EBook(...), LibraryItem): {isinstance(EBook('test', 'test'), LibraryItem)}")
print(f"issubclass(PDFPamphlet, LibraryItem): {issubclass(PDFPamphlet, LibraryItem)}")
print(f"isinstance(PDFPamphlet(...), LibraryItem): {isinstance(PDFPamphlet('test', 'test'), LibraryItem)}")

issubclass(PrintedBook, LibraryItem): True
isinstance(EBook(...), LibraryItem): True
issubclass(PDFPamphlet, LibraryItem): True
isinstance(PDFPamphlet(...), LibraryItem): True


## 19) Inventory report via abstraction

Write `summarize_items(items: list[LibraryItem]) -> list[str]` that uses only abstract methods/properties (`describe`, `media_type`, etc.). Demonstrate polymorphism by passing a mixed list of items.

In [21]:
def summarize_items(items: list[LibraryItem]) -> list[str]:
    return [f"{item.media_type}: {item.describe()}" for item in items]

mixed_items = [
    PrintedBook('123', 'Python Guide', 1),
    EBook('456', 'Digital Manual', 1, 20.0),
    AudioBook('789', 'Audio Course', 1, 300),
    PDFPamphlet('999', 'Quick Ref', 1)
]

summaries = summarize_items(mixed_items)
for summary in summaries:
    print(summary)

print: PrintedBook<123>: Python Guide (copies=1)
ebook: EBook<456>: Digital Manual (20.0 MB)
audiobook: AudioBook<789>: Audio Course (300 min)
pamphlet: PDFPamphlet<999>: Quick Ref


## 20) End-to-end scenario under ABC contract

Create a demo that:
- Builds a `Catalog` and `LoanDesk`
- Adds one of each concrete item and one registered virtual subclass instance
- Checks each out to a `Member`
- Prints a small receipt using `receipt_line()` and the computed due dates
Use only `LibraryItem`-level APIs at call sites (no `isinstance` branches).

In [23]:
def pdf_full_label(self) -> str:
    return f"[{self.media_type}] {self.title} — {self.isbn}"

PDFPamphlet.full_label = pdf_full_label

catalog = Catalog()
desk = LoanDesk(catalog)

items = [
    PrintedBook('PB001', 'Printed Guide', 2),
    EBook('EB001', 'Digital Manual', 1, 25.0),
    AudioBook('AB001', 'Audio Course', 1, 240),
    PDFPamphlet('PDF001', 'Quick Reference', 1)
]

for item in items:
    catalog.add_item(item)

member = Member('M001', 'user@example.com')

print("=== LIBRARY CHECKOUT DEMO ===")
for item in items:
    loan = desk.checkout(member, item)
    print(f"Receipt: {item.receipt_line()}")
    print(f"Due: {loan.due_date.strftime('%Y-%m-%d')}")
    print(f"Label: {item.full_label()}")
    print()

print(f"Total active loans: {len(desk.active_loans_for(member))}")

=== LIBRARY CHECKOUT DEMO ===
Receipt: PrintedBook<PB001>: Printed Guide (copies=1) | Loan period: 21 days
Due: 2025-12-07
Label: [print] Printed Guide — PB001

Receipt: EBook<EB001>: Digital Manual (25.0 MB) | Loan period: 14 days
Due: 2025-11-30
Label: [ebook] Digital Manual — EB001

Receipt: AudioBook<AB001>: Audio Course (240 min) | Loan period: 14 days
Due: 2025-11-30
Label: [audiobook] Audio Course — AB001

Receipt: PDFPamphlet<PDF001>: Quick Reference | Loan period: 7 days
Due: 2025-11-23
Label: [pamphlet] Quick Reference — PDF001

Total active loans: 4


## Python skills you'll need (Weeks 1–9)

- **Core syntax & data types:** variables, strings, numbers, booleans
- **Collections:** lists, dicts (basic use), simple comprehensions
- **Control flow:** `if/elif/else`, `for`, `while`
- **Functions & modules:** defining functions, parameters, returns, imports
- **File I/O & JSON (basic):** open/read/write, simple JSON usage
- **Classes & objects (Weeks 4–8):** classes, `__init__`, instance methods, overriding, `super()`
- **Encapsulation basics:** simple validation; naming conventions for "private" attributes
- **Error handling & testing (Week 7):** `try/except`, custom exceptions, basic `unittest`
- **Week 8 OOP:** single inheritance & polymorphism (no ABCs)
- **Week 9 focus:** **Abstract Base Classes (ABC)** with `abc.ABC` and `@abstractmethod`, abstract properties, class/instance abstract methods, virtual subclass **registration**, and light `typing.Protocol` usage (non-generic)
- **Standard library familiarity:** `abc`, `datetime`, built-in exceptions
